# 7주차 실습: 이진 탐색 트리(BST) 구현 및 성능 분석 — 학생용

## 목표·규약
재귀 탐색/삽입/삭제, 중위 정렬 검증, 삽입 순서별 높이·탐색 시간 비교.
Python 3.8+; 위에서 아래로 실행. 키는 비교 가능한 정수, 중복 삽입 무시, 없는 값 삭제는 변화 없음.
높이는 간선 기준(빈 트리 -1). 단순 BST는 균형을 보장하지 않습니다.
학생 TODO는 NotImplementedError로 즉시 중단하며 구현 후 테스트합니다.


In [37]:
import random, time, statistics
from collections import deque
print("환경 준비 완료")

환경 준비 완료


## Step 1. BSTNode
지난주 TreeNode의 data/left/right 구조를 재사용.

In [38]:
class BSTNode:
    def __init__(self, data):
        self.data = data
        self.left = None
        self.right = None
    def __repr__(self):
        return f"BSTNode({self.data!r})"

## Step 2–3. 전체 BST 클래스
재귀 결과를 `node.left/right`, 최상위 결과를 `self.root`에 재연결하세요.
삭제: 리프→None / 자식 하나→자식 / 둘→오른쪽 최소값 복사 후 후속자 삭제.
후속자는 오른쪽 자식을 가질 수 있어 재연결이 필요합니다.
하나의 클래스에 전체 메서드를 넣어 동일 이름 자기 상속을 피했습니다.
성능 실험용 반복 메서드도 포함합니다. 큰 편향 트리에 재귀 메서드를 호출하지 마세요.


In [39]:
class BinarySearchTree:
    def __init__(self):
        self.root = None

    def search(self, key):
        return self._search(self.root, key)

    def _search(self, node, key):
        if node is None:
          return False
        if key == node.data:
          return True
        if key < node.data:
          return self._search(node.left, key)
        else:
          return self._search(node.right, key)

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
          return BSTNode(key)

        if key < node.data:
          node.left = self._insert(node.left, key)
        elif key > node.data:
          node.right = self._insert(node.right, key)

        return node

    def delete(self, key):
        self.root = self._delete(self.root, key)

    def _delete(self, node, key):
        if node is None:
          return None

        if key < node.data:
          node.left = self._delete(node.left, key)
        elif key > node.data:
          node.right = self._delete(node.right, key)
        else:
          if node.left is None:
            return node.right
          elif node.right is None:
            return node.left

          successor = self._find_min(node.right)
          node.data = successor.data
          node.right = self._delete(node.right, successor.data)

        return node

    def _find_min(self, node):
        current = node
        while current.left is not None:
          current = current.left
        return current

    def inorder(self):
        result = []
        def _inorder(node):
          if node:
            _inorder(node.left)
            result.append(node.data)
            _inorder(node.right)
        _inorder(self.root)
        return result

    def height(self):
        def _height(node):
          if node is None:
            return -1
          return 1+ max(_height(node.left), _height(node.right))
        return _height(self.root)

    def count_nodes(self):
        def _count(node):
          if node is None:
            return 0
          return 1+_count(node.left) + _count(node.right)
        return _count(self.root)

    def insert_iterative(self, key):
        if self.root is None:
          self.root = BSTNode(key)
          return

        current = self.root
        while True:
          if key == current.data:
            return
          elif key < current.data:
            if current.left is None:
              current.left = BSTNode(key)
              return
            current = current.left
          else:
            if current.right is None:
              current.right = BSTNode(key)
              return
            current = current.right

    def search_iterative(self, key):
        current = self.root
        while current is not None:
          if key == current.data:
            return True
          if key < current.data:
            current = current.left
          else:
            current = current.right
        return False

    def height_iterative(self):
        if self.root is None:
          return -1

        stack = [(self.root, 0)]
        max_height = -1

        while stack:
          node, depth = stack.pop()
          max_height=max(max_height, depth)

          if node.left:
            stack.append((node.left, depth +1))
          if node.right:
            stack.append((node.right, depth +1))

        return max_height



## Step 4. 삽입/삭제 로직 검증
후속자의 오른쪽 자식, 루트 삭제, 중복, 없는 값, 무작위 혼합 연산까지 검사.

In [40]:
def make_bst(values, iterative=False):
    tree = BinarySearchTree()
    for value in values:
        if iterative: tree.insert_iterative(value)
        else: tree.insert(value)
    return tree

def assert_bst(tree, expected):
    # 조상 범위까지 검사: 바로 위 부모만 비교하는 오류 방지
    def valid(node, low=None, high=None):
        if node is None: return True
        if low is not None and node.data <= low: return False
        if high is not None and node.data >= high: return False
        return valid(node.left, low, node.data) and valid(node.right, node.data, high)
    assert valid(tree.root)
    assert tree.inorder() == sorted(set(expected))
    assert tree.count_nodes() == len(set(expected))


In [41]:
values = [50, 30, 70, 20, 40, 60, 80]
bst = make_bst(values)
assert_bst(bst, values)
assert bst.search(40) is True and bst.search(99) is False
for key in [20, 30, 70]:
    bst.delete(key)
    values.remove(key)
    assert_bst(bst, values)
    assert not bst.search(key)
    print("삭제", key, bst.inorder())
# 루트 두 자식 삭제
bst.delete(50); values.remove(50)
assert_bst(bst, values)
assert bst.root.data == 60

empty = BinarySearchTree()
empty.delete(1)
assert_bst(empty, [])
assert empty.height() == -1 and not empty.search(1)
for vals in [[5], [5,3], [5,7]]:
    t = make_bst(vals)
    t.delete(5)
    assert_bst(t, vals[1:])
# 후속자 60은 오른쪽 자식65를 갖는다: 65가 손실되면 안 됨.
t = make_bst([50,30,70,60,65,80])
t.delete(50)
assert t.root.data == 60 and t.root.right.left.data == 65
assert_bst(t, [30,60,65,70,80])
t.insert(65); t.delete(999)
assert_bst(t, [30,60,65,70,80])
# 전체 삭제로 빈 트리 복귀
for key in [30,60,65,70,80]: t.delete(key)
assert_bst(t, [])
# 고정 시드의 혼합 삽입/삭제를 Python set과 대조
rng = random.Random(7)
t, reference = BinarySearchTree(), set()
for _ in range(200):
    key = rng.randrange(50)
    if rng.random() < 0.6: t.insert(key); reference.add(key)
    else: t.delete(key); reference.discard(key)
    assert_bst(t, reference)
    assert t.search(key) == (key in reference)
print("모든 삽입·삭제·경계·불변식 테스트 통과")


삭제 20 [30, 40, 50, 60, 70, 80]
삭제 30 [40, 50, 60, 70, 80]
삭제 70 [40, 50, 60, 80]
모든 삽입·삭제·경계·불변식 테스트 통과


## Step 5. 높이 비교 실험
n=1000 재귀 삽입은 편향 트리에서 RecursionError가 날 수 있습니다. 반복 구현을 이용하며 재귀 제한을 올리지 않습니다.
최소 가능 높이 n>=1: ceil(log2(n+1))-1 = n.bit_length()-1.
무작위 트리는 기대 높이 O(log n)이지만 매번 균형이 보장되는 것은 아닙니다.


In [42]:
def build_for_experiment(n, order="random", seed=7):
    values = list(range(n))

    if order == "random":
      rng = random.Random(seed)
      rng.shuffle(values)

    tree = BinarySearchTree()
    for value in values:
      tree.insert_iterative(value)

    return tree

def measure_height(n, order="random", seed=7):
    return build_for_experiment(n, order, seed).height_iterative()

n = 1000
bst_random = build_for_experiment(n, "random")
bst_sorted = build_for_experiment(n, "sorted")
assert bst_sorted.height_iterative() == n-1
print("최소 가능 높이:", n.bit_length()-1)
print("무작위/정렬 높이:", bst_random.height_iterative(), bst_sorted.height_iterative())


최소 가능 높이: 9
무작위/정렬 높이: 24 999


## Step 6. 탐색 시간 측정
동일한 키 목록, 반복 탐색 구현, 구성 시간 제외, 워밍업, 5회 중앙값. 측정값은 환경에 따라 달라집니다.

In [50]:
targets = [random.Random(17+i).randrange(n) for i in range(1000)]
def timed_search(tree, targets, repeats=5):
    for target in targets:
      tree.search_iterative(target)

    times = []
    for _ in range(repeats):
      start = time.perf_counter()
      for target in targets:
        tree.search_iterative(target)
      end = time.perf_counter()
      times.append(end - start)

    return statistics.median(times)

random_time = timed_search(bst_random, targets)
sorted_time = timed_search(bst_sorted, targets)
print(f"무작위 트리 탐색 중앙값: {random_time:.6f}s")
print(f"정렬 삽입 트리 중앙값: {sorted_time:.6f}s")
if random_time > 0: print(f"측정 시간비: {sorted_time/random_time:.2f}")

무작위 트리 탐색 중앙값: 0.001142s
정렬 삽입 트리 중앙값: 0.033262s
측정 시간비: 29.12


## 과제: BST 구현 및 성능 분석
1. 모든 로직 테스트 통과. 2. n=100/1000/10000의 두 삽입 순서 높이를 표로 정리.
3. 최소 높이와 비교하고 성능 영향을 3~5문장으로 분석. 시간 측정 조건도 명시.
4. 개인 GitHub Week7 폴더, 주당 최소 2회 커밋, 과제 폼 GitHub 링크 제출.
**주의:** 정렬 순서 실제 삽입은 전체 O(n²)입니다. n=10000 셀은 느릴 수 있어 마지막에 실행합니다. 체인 직접 구성은 같은 삽입 실험이 아니므로 대체하지 않습니다.


In [44]:
rows = []
for size in [100, 1000, 10000]:
    h_random = measure_height(size, "random")
    h_sorted = measure_height(size, "sorted")
    assert h_sorted == size - 1
    rows.append((size, size.bit_length()-1, h_random, h_sorted))
print("n | 최소 가능 높이 | 무작위 높이 | 정렬 높이")
for row in rows: print(*row, sep=" | ")


n | 최소 가능 높이 | 무작위 높이 | 정렬 높이
100 | 6 | 12 | 99
1000 | 9 | 24 | 999
10000 | 13 | 29 | 9999


## 분석 작성란
- 무작위/정렬 입력에서 높이:
실험 결과, 데이터 10,000개 기준 정렬된 입력의 트리 높이는 9999로 데이터 수(n-1)에 비례하여 최대 깊이로 증가했습니다. 반면 무작위 순서로 삽입한 트리의 높이는 29로 그 증가폭이 크게 억제되었습니다. 이는 입력 데이터가 정렬되어 있을 때 노드가 한쪽으로만 깊게 뻗어 선형적인 형태O(n)가 됨을 보여줍니다. 무작위 입력은 데이터를 양쪽 서브트리로 분산시켜 O(log n) 수준의 낮은 트리 높이를 형성하게 만듭니다.
- 동일 탐색 대상의 시간 중앙값과 실험 조건:
시간 측정은 1,000개의 무작위 타겟을 반복 탐색하는 과정을 5회 반복하여 캐시 영향을 통제하고 중앙값을 구하는 조건으로 진행되었습니다. 측정 결과 무작위 트리는 약 0.0011초, 정렬된 편향 트리는 약 0.033초가 소요되어 정렬 삽입 트리의 탐색 시간이 30배 이상 오래 걸렸습니다. 이는 탐색 성능O(h)이 트리의 높이에 절대적으로 의존한다는 것을 명확히 보여줍니다. 정렬된 트리는 사실상 연결 리스트와 같이 동작하므로, 값을 찾기 위해 깊은 노드까지 모두 순회해야 해 심각한 성능 저하가 발생합니다.
- 편향 원인과 균형 보장 여부:
트리가 편향되는 원인은 오름차순과 같이 정렬된 데이터가 연속해서 삽입될 때, 항상 이전 노드보다 큰 값만 들어와 오른쪽 자식으로만 계속 연결되기 때문입니다. 기본 이진 탐색 트리(BST)는 노드를 삽입할 때 트리의 구조를 재조정하는 로직이 없어서 이러한 구조적 악화를 막지 못합니다. 셔플링(무작위 섞기)을 통해 확률적으로 편향을 피할 수는 있지만, 어떤 순서의 입력이 들어오더라도 균형이 유지된다고 시스템적으로 보장하지는 않습니다. 항상 최적의 균형을 보장하려면 삽입/삭제 시 회전 연산을 수행하는 AVL 트리나 레드-블랙 트리 등을 사용해야 합니다.
- 최소 높이와 무작위 높이가 다른 이유:
트리의 최소 가능 높이는 모든 노드가 양쪽으로 빈틈없이 꽉 채워진 '완전 이진 트리' 형태일 때만 달성되며, 10,000개 데이터 기준 13입니다. 데이터를 무작위 순서로 삽입하더라도, 삽입되는 값들이 항상 정중앙을 가르는 이상적인 중앙값이 아니기 때문에 완벽한 대칭 구조가 만들어지지는 않습니다. 특정 구간에서는 여전히 작은 편향들이 발생하여 일부 서브트리가 길어지므로 무작위 트리(높이 29)가 최소 높이보다 더 높게 측정됩니다. 즉, 무작위 삽입은 트리를 적당히 납작한 O(log n) 형태로 유지해주지만, 수학적으로 가능한 최상의 빽빽한 형태까지 도달하지는 못하기 때문에 높이 차이가 발생합니다.

이번 실험의 탐색 시간 측정은 1,000개의 무작위 타겟을 워밍업 후 5회 반복 탐색하여 중앙값을 도출하는 조건으로 진행되었습니다. 측정 결과, 데이터 10,000개 기준 최소 가능 높이(13)에 비교적 근접한 높이(29)를 기록한 무작위 삽입 트리는 탐색에 약 0.0011초가 소요되어 우수한 성능을 보였습니다. 반면, 데이터가 정렬된 채로 삽입되어 최대 높이(9999)에 도달한 편향 트리는 무작위 트리보다 탐색 시간이 약 30배나 느린 0.033초가 소요되었습니다. 결론적으로 이진 탐색 트리의 탐색 성능은 트리의 높이에 절대적으로 의존하므로, 높이가 최소 수준으로 억제되지 못하고 선형적으로 깊어질 경우 O(n)의 탐색 시간으로 인해 심각한 성능 저하가 발생함을 확인했습니다.

## 복잡도 정리
search/insert/delete O(h+1), 재귀 보조 공간 O(h+1). 반복 탐색/삽입 공간 O(1).
중위 순회 O(n) 시간, O(h+1) 호출 스택 + O(n) 출력. 높이 계산 O(n).
균형 높이 O(log n), 최악 O(n); 서로 다른 키의 무작위 순열 삽입은 기대 O(log n) 연산.
Tree Sort 기대 O(n log n), 최악 O(n²); 중복을 보존하려면 카운터 등 정책 변경 필요.


## 선택 심화·시각화
- VisuAlgo BST 모드에서 50,30,70,20,40,60,80 삽입 후 20,30,70 삭제와 코드 결과 비교.
- 중위 선행자를 이용한 삭제로 변경.
- 셔플 전처리는 기대 성능 개선이지 최악 균형 보장이 아님을 설명.
- 자손 수 저장으로 kth_smallest 구현: O(h), 균형일 때만 O(log n). 삽입/삭제마다 size 갱신 필요.
- 다음 주 중간 평가: 연결리스트, 스택/큐, 트리, BST 복습.
